In [0]:
caminho_base = "/Volumes/mvp_tarifas_energia/bronze/dados_brutos_aneel/"

df_tarifas = spark.read.option("header", "true") \
                        .option("sep", ",") \
                        .option("encoding", "UTF-8") \
                        .option("quote", "\"") \
                        .option("escape", "\"") \
                        .csv(caminho_base + "tarifas_por_distribuidora.csv")

print("Colunas:", df_tarifas.columns)
print("Total de linhas:", df_tarifas.count())
display(df_tarifas.limit(10))

In [0]:
df_bandeira_adicional = spark.read.option("header", "true") \
                                    .option("sep", ";") \
                                    .option("encoding", "UTF-8") \
                                    .option("quote", "\"") \
                                    .csv(caminho_base + "bandeira-tarifaria-adicional.csv")

print("--- Bandeira Adicional ---")
print("Colunas:", df_bandeira_adicional.columns)
display(df_bandeira_adicional.limit(5))


df_bandeira_acionamento = spark.read.option("header", "true") \
                                      .option("sep", ";") \
                                      .option("encoding", "UTF-8") \
                                      .option("quote", "\"") \
                                      .csv(caminho_base + "bandeira-tarifaria-acionamento.csv")

print("--- Bandeira Acionamento ---")
print("Colunas:", df_bandeira_acionamento.columns)
display(df_bandeira_acionamento.limit(5))

In [0]:
import zipfile
import shutil
import os

caminho_zip = "/Volumes/mvp_tarifas_energia/bronze/dados_brutos_aneel/reclamacoes-n1e2-distribuidoras-2025.csv"
caminho_extracao = "/tmp/reclamacoes_extraido/"

shutil.copy(caminho_zip, "/tmp/reclamacoes.zip")

with zipfile.ZipFile("/tmp/reclamacoes.zip", 'r') as z:
    print("Arquivos dentro do zip:", z.namelist())
    z.extractall(caminho_extracao)

nome_csv_real = [f for f in os.listdir(caminho_extracao) if f.endswith(".csv")][0]
shutil.copy(caminho_extracao + nome_csv_real, 
            "/Volumes/mvp_tarifas_energia/bronze/dados_brutos_aneel/" + nome_csv_real)

print("Arquivo extraído e salvo como:", nome_csv_real)

In [0]:
df_reclamacoes = spark.read.option("header", "true") \
                             .option("sep", ";") \
                             .option("encoding", "UTF-8") \
                             .option("quote", "\"") \
                             .csv(caminho_base + "manifestacoes-1-2-niveis-distribuidora-2025.csv")

print("--- Reclamações ---")
print("Colunas:", df_reclamacoes.columns)
print("Total de linhas:", df_reclamacoes.count())
display(df_reclamacoes.limit(5))

In [0]:
df_cadastro_distribuidoras = df_reclamacoes.select(
    "SigAgente", "SigUF", "SigRegiao", "NomClassificacaoAgente"
).distinct()

df_cadastro_distribuidoras.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_tarifas_energia.silver.dim_distribuidora")

print("Tabela dim_distribuidora salva com sucesso!")
display(spark.sql("SELECT * FROM mvp_tarifas_energia.silver.dim_distribuidora LIMIT 5"))

In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date

df_tarifas_silver = df_tarifas.select(
    col("SigAgente"),
    col("NumCNPJDistribuidora"),
    to_date(col("DatInicioVigencia"), "yyyy-MM-dd").alias("DatInicioVigencia"),
    to_date(col("DatFimVigencia"), "yyyy-MM-dd").alias("DatFimVigencia"),
    col("DscBaseTarifaria"),
    col("DscSubGrupo"),
    col("DscModalidadeTarifaria"),
    col("DscClasse"),
    col("DscSubClasse"),
    col("DscDetalhe"),
    col("NomPostoTarifario"),
    col("DscUnidadeTerciaria"),
    regexp_replace(col("VlrTUSD"), ",", ".").cast("double").alias("VlrTUSD"),
    regexp_replace(col("VlrTE"), ",", ".").cast("double").alias("VlrTE")
)

df_tarifas_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_tarifas_energia.silver.fato_tarifas")

print("Tabela fato_tarifas salva com sucesso!")
display(spark.sql("SELECT * FROM mvp_tarifas_energia.silver.fato_tarifas LIMIT 5"))

In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date

df_bandeira_adicional_silver = df_bandeira_adicional.select(
    to_date(col("DatVigencia"), "yyyy-MM-dd").alias("DatVigencia"),
    col("DscResolucao"),
    col("NomBandeiraAcionada"),
    regexp_replace(col("VlrAdicionalBandeiraRSMWh"), ",", ".").cast("double").alias("VlrAdicionalBandeiraRSMWh")
)

df_bandeira_adicional_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_tarifas_energia.silver.dim_bandeira_adicional")

print("--- Bandeira Adicional salva ---")


df_bandeira_acionamento_silver = df_bandeira_acionamento.select(
    to_date(col("DatCompetencia"), "yyyy-MM-dd").alias("DatCompetencia"),
    col("NomBandeiraAcionada"),
    regexp_replace(col("VlrAdicionalBandeira"), ",", ".").cast("double").alias("VlrAdicionalBandeira")
)

df_bandeira_acionamento_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_tarifas_energia.silver.fato_bandeira_acionamento")

print("--- Bandeira Acionamento salva ---")

In [0]:
from pyspark.sql.functions import col, when, upper, trim, translate, regexp_replace

def normalizar_nome(coluna):
    # Só trata acento, maiúsculo/minúsculo e hífen/espaço extra —
    # NUNCA remove palavras ou sufixos por regex, para evitar colisão entre nomes diferentes
    sem_acento = translate(coluna, "ÁÀÂÃÄáàâãäÉÈÊËéèêëÍÌÎÏíìîïÓÒÔÕÖóòôõöÚÙÛÜúùûüÇç", 
                                   "AAAAAaaaaaEEEEeeeeIIIIiiiiOOOOOooooUUUUuuuuCc")
    return trim(regexp_replace(upper(sem_acento), r"[\s\-]+", " "))

# De-para ampliado: casos de fusão/marca diferente + casos de sufixo -D/-DIS/-DISTRIBUI
# que antes a normalização tentava resolver sozinha (e causava colisão)
de_para_distribuidoras = [
    ("COELBA", "Neoenergia Coelba"),
    ("NEOENERGIA PE", "Neoenergia Pernambuco"),
    ("COSERN", "Neoenergia Cosern"),
    ("ELEKTRO", "Neoenergia Elektro"),
    ("CEA", "CEA Equatorial"),
    ("CEEE", "CEEE Equatorial"),
    ("CEEE-D", "CEEE Equatorial"),
    ("ELETROPAULO", "Enel SP"),
    ("RGE SUL", "RGE"),
    ("AMBAR AMAZONAS", "Amazonas Energia"),
    ("AMBAR ENERGIA RR", "Roraima Energia"),
    ("LIGHT SESA", "Light"),
    ("CPFL JAGUARI", "CPFL Paulista"),
    ("CPFL LESTE PAULI", "CPFL Paulista"),
    ("CPFL SUL PAULIST", "CPFL Paulista"),
    ("CPFL MOCOCA", "CPFL Paulista"),
    ("CPFL-PAULISTA", "CPFL Paulista"),
    ("CPFL-PIRATINING", "CPFL Piratininga"),
    ("CPFL SANTA CRUZ", "CPFL Paulista"),
    ("EAC", "Energisa AC"),
    ("EMR", "Energisa Minas-Rio"),
    ("EMS", "Energisa MS"),
    ("EMT", "Energisa MT"),
    ("EPB", "Energisa PB"),
    ("ERO", "Energisa RO"),
    ("ESE", "Energisa SE"),
    ("ESS", "Energisa Sul-Sudeste"),
    ("ETO", "Energisa TO"),
    ("CFLO", "Energisa Sul-Sudeste"),
    ("CNEE", "Energisa Sul-Sudeste"),
    ("EDEVP", "Energisa Sul-Sudeste"),
    ("EEB", "Energisa Sul-Sudeste"),
    ("EFLUL", "Energisa Sul-Sudeste"),
    ("COOPERALIANCA", "Cooperaliança"),
    ("CERAL ANITAPOLIS", "Ceral Anitápolis"),
    ("CERAL ARARUAMA", "Ceral Araruama"),
    ("CERAL-DIS", "Ceral"),
    ("CERMISSOES", "Cermissões"),
    ("CASTRO-DIS", "Castro"),
    ("CEMIG-D", "Cemig"),
    ("COPEL-DIS", "Copel"),
    ("CRELUZ-D", "Creluz"),
    ("CERSAD DISTRIBUI", "Cersad"),
    ("ENEL CE", "Enel CE"),
    ("ENEL RJ", "Enel RJ"),
    ("EQUATORIAL AL", "Equatorial AL"),
    ("EQUATORIAL GO", "Equatorial GO"),
    ("EQUATORIAL MA", "Equatorial MA"),
    ("EQUATORIAL PA", "Equatorial PA"),
    ("EQUATORIAL PI", "Equatorial PI"),
]

df_de_para = spark.createDataFrame(de_para_distribuidoras, ["SigAgenteOriginal", "SigAgenteCorrigido"])

In [0]:
from pyspark.sql.functions import col, when

# 1. Aplica o de-para (usando normalização só para casar, preservando o valor original)
df_de_para_norm = df_de_para.withColumn("ChaveOriginal", normalizar_nome(col("SigAgenteOriginal")))

df_tarifas_corrigida = spark.table("mvp_tarifas_energia.silver.fato_tarifas") \
    .withColumn("ChaveNormalizada", normalizar_nome(col("SigAgente"))) \
    .join(df_de_para_norm, col("ChaveNormalizada") == col("ChaveOriginal"), how="left") \
    .withColumn("SigAgenteJoin", when(col("SigAgenteCorrigido").isNotNull(), normalizar_nome(col("SigAgenteCorrigido"))).otherwise(col("ChaveNormalizada"))) \
    .drop("ChaveNormalizada", "ChaveOriginal", "SigAgenteOriginal", "SigAgenteCorrigido")

df_dim_distribuidora = spark.table("mvp_tarifas_energia.silver.dim_distribuidora") \
    .withColumnRenamed("SigAgente", "SigAgenteCadastro") \
    .withColumn("ChaveCadastro", normalizar_nome(col("SigAgenteCadastro")))

# 2. IMPORTANTE: confere se há chaves de cadastro duplicadas ANTES do join
#    (se duas linhas do cadastro normalizarem para a mesma chave, o join vai multiplicar)
duplicatas_cadastro = df_dim_distribuidora.groupBy("ChaveCadastro").count().filter(col("count") > 1)
qtd_duplicatas = duplicatas_cadastro.count()
print("Chaves de cadastro duplicadas (risco de multiplicação no JOIN):", qtd_duplicatas)
if qtd_duplicatas > 0:
    display(duplicatas_cadastro)

# 3. Faz o JOIN
df_gold_tarifas = df_tarifas_corrigida.join(
    df_dim_distribuidora,
    df_tarifas_corrigida["SigAgenteJoin"] == df_dim_distribuidora["ChaveCadastro"],
    how="left"
).drop("SigAgenteJoin", "SigAgenteCadastro", "ChaveCadastro")

# 4. VERIFICAÇÃO ANTI-DUPLICAÇÃO: a Gold não pode ter mais linhas que a Silver original
qtd_silver = spark.table("mvp_tarifas_energia.silver.fato_tarifas").count()
qtd_gold = df_gold_tarifas.count()
print(f"Linhas na Silver (fato_tarifas): {qtd_silver}")
print(f"Linhas na Gold (após JOIN): {qtd_gold}")
if qtd_gold != qtd_silver:
    print("⚠️ ALERTA: quantidade de linhas mudou após o JOIN — há duplicação ou perda de dados!")
else:
    print("✅ OK: quantidade de linhas preservada, sem duplicação.")

# 5. Checagem de correspondência (como antes)
sem_correspondencia_final = df_gold_tarifas.filter(col("SigUF").isNull()).select("SigAgente").distinct()
print("Distribuidoras sem correspondência:", sem_correspondencia_final.count())
display(sem_correspondencia_final.orderBy("SigAgente"))

# 6. Só salva se passou na verificação de integridade
if qtd_gold == qtd_silver:
    df_gold_tarifas.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("mvp_tarifas_energia.gold.fato_tarifas_enriquecida")
    print("--- Gold: fato_tarifas_enriquecida salva ---")
else:
    print("--- Gold NÃO salva — corrija a duplicação antes de prosseguir ---")

In [0]:
from pyspark.sql.functions import col, lead, date_trunc
from pyspark.sql.window import Window

janela = Window.partitionBy("NomBandeiraAcionada").orderBy("DatVigencia")

df_bandeira_com_fim_vigencia = spark.table("mvp_tarifas_energia.silver.dim_bandeira_adicional") \
    .withColumn("MesVigencia", date_trunc("month", col("DatVigencia"))) \
    .withColumn("MesFimVigencia", lead("MesVigencia").over(janela))

df_bandeira_gold = spark.table("mvp_tarifas_energia.silver.fato_bandeira_acionamento").alias("acion") \
    .withColumn("MesCompetencia", date_trunc("month", col("DatCompetencia"))) \
    .join(
        df_bandeira_com_fim_vigencia.alias("adic"),
        (col("acion.NomBandeiraAcionada") == col("adic.NomBandeiraAcionada")) &
        (col("MesCompetencia") >= col("adic.MesVigencia")) &
        (
            col("adic.MesFimVigencia").isNull() |
            (col("MesCompetencia") < col("adic.MesFimVigencia"))
        ),
        how="left"
    ) \
    .select(
        col("acion.DatCompetencia"),
        col("acion.NomBandeiraAcionada"),
        col("acion.VlrAdicionalBandeira").alias("VlrAdicionalAplicadoNoMes"),
        col("adic.VlrAdicionalBandeiraRSMWh").alias("VlrHomologadoNaResolucao"),
        col("adic.DscResolucao")
    )

print("Total de linhas:", df_bandeira_gold.count())
display(df_bandeira_gold.orderBy("DatCompetencia"))

df_bandeira_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_tarifas_energia.gold.fato_bandeira_completa")

print("--- Gold: fato_bandeira_completa salva ---")

In [0]:
%sql
SELECT COUNT(*) as total_gold FROM mvp_tarifas_energia.gold.fato_bandeira_completa;
SELECT COUNT(*) as total_silver FROM mvp_tarifas_energia.silver.fato_bandeira_acionamento;